# 03 – Classical Machine Learning Baselines

**Purpose:** Train, evaluate, and compare TF-IDF classical ML baseline models (Logistic Regression, Linear SVM, and Multinomial Naive Bayes) on the customer support dataset.

This notebook demonstrates:
1. Loading the preprocessed training and test splits.
2. Training each baseline model using standard scikit-learn Pipelines.
3. Running evaluation to collect key performance indicators: Accuracy, Precision, Recall, F1-score, Training Time, Inference Latency, and Model Size on disk.
4. Visualizing confusion matrices side-by-side.
5. Exporting predictions, reports, and metrics.

## 0. Environment Setup

Import baseline modules and register repository root path.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import IPython.display as display

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

## 1. Trigger Baseline Models Pipeline

We can trigger training and evaluation programmatically via the baseline CLI layer.

In [ ]:
from src.models.baselines.cli import train_baselines, evaluate_baselines

# Run training on train split
train_baselines()

# Run evaluation on test split
evaluate_baselines()

## 2. Model Performance Comparison

Load the saved `metrics.json` file containing all metrics and display it as a comparison table.

In [ ]:
metrics_path = REPO_ROOT / "outputs" / "metrics" / "metrics.json"
with open(metrics_path, encoding="utf-8") as f:
    metrics_dict = json.load(f)

# Convert to pandas DataFrame and transpose for clean view
df_comparison = pd.DataFrame(metrics_dict).T

# Reorder columns for readability
cols = [
    "accuracy",
    "f1_weighted",
    "precision_weighted",
    "recall_weighted",
    "training_time_seconds",
    "prediction_latency_ms_per_sample",
    "model_size_bytes",
]
df_comparison = df_comparison[cols]
df_comparison

## 3. Visualizing Metrics

Compare Model Accuracy and Prediction Latency side-by-side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Barplot
df_comparison["accuracy"].plot(kind="bar", ax=axes[0], color=["#3498db", "#2ecc71", "#e74c3c"])
axes[0].set_title("Model Accuracy Comparison")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0.7, 1.0)
axes[0].grid(axis="y", linestyle="--", alpha=0.7)
axes[0].tick_params(axis="x", rotation=15)

# Latency Barplot
df_comparison["prediction_latency_ms_per_sample"].plot(kind="bar", ax=axes[1], color=["#34495e", "#9b59b6", "#f1c40f"])
axes[1].set_title("Prediction Latency (ms per sample)")
axes[1].set_ylabel("Latency (ms)")
axes[1].grid(axis="y", linestyle="--", alpha=0.7)
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 4. Render Confusion Matrices

Display the generated side-by-side confusion matrix plot.

In [ ]:
display.Image(filename=str(REPO_ROOT / "outputs" / "figures" / "confusion_matrix.png"))

## 5. Classification Reports

Print the classification report text file.

In [ ]:
report_path = REPO_ROOT / "outputs" / "metrics" / "classification_report.txt"
with open(report_path, encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Export Phase Manifest
from src.utils.artifacts import save_yaml
from src.api.app import get_git_commit

manifest = {
    "phase": "03_Baseline_Models",
    "metrics": metrics_dict,
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_03_baselines.yaml")
print("YAML manifest saved successfully:")
print(manifest)
